In [1]:
pip install pydicom

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas 
import numpy,os
import pydicom,cv2

label=pandas.read_csv("/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train_label_coordinates.csv")
train=pandas.read_csv("/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train.csv")

print(numpy.array(label["study_id"]),numpy.array(label["series_id"]))
os.mkdir(f"/kaggle/working/train_wolf/")

for i in zip(numpy.array(label["study_id"]),numpy.array(label["series_id"])
	,numpy.array(label['instance_number']),numpy.array(label['condition']),numpy.array(label['level']),range(len(numpy.array(label['level'])))):
	
	w_1,w_2='_'.join(i[3].split(' ')).lower(),'_'.join(i[4].split('/')).lower()
	w_=f"{'_'.join(i[3].split(' '))}_{'_'.join(i[4].split('/'))}".lower()
	wolf=train[train["study_id"]==i[0]].to_dict('list')[w_][0]
	if wolf=="Normal/Mild":
		wolf="normal"


	if wolf=="normal" or wolf=="Moderate" or wolf=="Severe":
		if os.path.exists(f"/kaggle/working/train_wolf/{w_1}")==False:
			os.mkdir(f"/kaggle/working/train_wolf/{w_1}")
			os.mkdir(f"/kaggle/working/train_wolf/{w_1}/{w_}_{wolf}")
			cv2.imwrite(f"/kaggle/working/train_wolf/{w_1}/{w_}_{wolf}/{i[5]}_{w_}.jpg",cv2.resize(cv2.normalize(pydicom.dcmread(f"/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train_images/{i[0]}/{i[1]}/{i[2]}.dcm").pixel_array, None, 0, 255, cv2.NORM_MINMAX).astype('uint8'),(512,512)))

		else:
			if os.path.exists(f"/kaggle/working/train_wolf/{w_1}/{w_}_{wolf}")==False:
				os.mkdir(f"/kaggle/working/train_wolf/{w_1}/{w_}_{wolf}")
				cv2.imwrite(f"/kaggle/working/train_wolf/{w_1}/{w_}_{wolf}/{i[5]}_{w_}.jpg",cv2.resize(cv2.normalize(pydicom.dcmread(f"/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train_images/{i[0]}/{i[1]}/{i[2]}.dcm").pixel_array, None, 0, 255, cv2.NORM_MINMAX).astype('uint8'),(512,512)))
			else:
				cv2.imwrite(f"/kaggle/working/train_wolf/{w_1}/{w_}_{wolf}/{i[5]}_{w_}.jpg",cv2.resize(cv2.normalize(pydicom.dcmread(f"/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification/train_images/{i[0]}/{i[1]}/{i[2]}.dcm").pixel_array, None, 0, 255, cv2.NORM_MINMAX).astype('uint8'),(512,512)))
	else:
		print(w_,w_1,w_2,wolf)
	#print(w_,w_1,w_2,wolf)


US1_J2KR.dcm:   0%|          | 38.0/154k [00:00<01:12, 2.13kB/s]
MR-SIEMENS-DICOM-WithOverlays.dcm:   0%|          | 125/511k [00:00<01:48, 4.73kB/s]
OBXXXX1A.dcm:   0%|          | 119/486k [00:00<01:51, 4.35kB/s]
US1_UNCR.dcm:   0%|          | 226/923k [00:00<02:36, 5.88kB/s]
color3d_jpeg_baseline.dcm:   0%|          | 1.50k/6.14M [00:00<05:16, 19.4kB/s]


[   4003253    4003253    4003253 ... 4290709089 4290709089 4290709089] [ 702807833  702807833  702807833 ... 4237840455 4237840455 4237840455]
left_subarticular_stenosis_l1_l2 left_subarticular_stenosis l1_l2 nan
left_subarticular_stenosis_l2_l3 left_subarticular_stenosis l2_l3 nan
left_subarticular_stenosis_l3_l4 left_subarticular_stenosis l3_l4 nan
left_subarticular_stenosis_l4_l5 left_subarticular_stenosis l4_l5 nan
left_subarticular_stenosis_l5_s1 left_subarticular_stenosis l5_s1 nan
right_neural_foraminal_narrowing_l1_l2 right_neural_foraminal_narrowing l1_l2 nan
right_neural_foraminal_narrowing_l2_l3 right_neural_foraminal_narrowing l2_l3 nan
right_neural_foraminal_narrowing_l3_l4 right_neural_foraminal_narrowing l3_l4 nan
right_neural_foraminal_narrowing_l4_l5 right_neural_foraminal_narrowing l4_l5 nan
right_neural_foraminal_narrowing_l5_s1 right_neural_foraminal_narrowing l5_s1 nan
right_neural_foraminal_narrowing_l5_s1 right_neural_foraminal_narrowing l5_s1 nan
right_neural_f

In [40]:
import argparse
import os
import random,numpy
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [41]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [42]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

workers = 2
batch_size = 10
nz = 100
num_epochs = 5
lr = 0.001
beta1 = 0.5
ngpu=1
ngf,nc = 3,3
ndf = 64

Random Seed:  999


In [43]:
transform = transforms.Compose(
    [transforms.Resize(256),transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.ImageFolder(root=f'/kaggle/working/train_wolf/right_neural_foraminal_narrowing',transform=transform)
dataloader = torch.utils.data.DataLoader(trainset,batch_size=batch_size,shuffle=True,num_workers=2)

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [44]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.devil=nn.Sequential(
            nn.Conv2d(3, 4,3),
            nn.BatchNorm2d(4),
            nn.ReLU(),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(4, 8, 3),
            nn.BatchNorm2d(8),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(8, 16, 3),
            nn.BatchNorm2d(16),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(16, 32, 3),
            nn.BatchNorm2d(32),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(32, 64, 3),
            nn.BatchNorm2d(64),
            nn.AvgPool2d(2, 2),

            nn.Conv2d(64, 128, 3),
            nn.BatchNorm2d(128),
            nn.AvgPool2d(2, 2),

            nn.Flatten(),

            nn.Linear(512, 264),
            nn.ReLU(),
            nn.Linear(264, 132),
            nn.ReLU(),
            nn.Linear(132, 64),
            nn.ReLU(),
            nn.Linear(64, 15)
        )

    def forward(self,x):
        return self.devil(x)


In [45]:
netD = Discriminator().to(device)
if (device.type == 'cuda') and (ngpu > 1):
    netD = nn.DataParallel(netD, list(range(ngpu)))
netD.apply(weights_init)

unique, counts = np.unique(np.array(trainset.targets), return_counts=True)
print("Unique:",unique,counts)
class_counts = dict(zip(unique, counts))
weights=[]
for i in range(len(dataloader.dataset.classes)):
    weights.append(max(class_counts.values())/class_counts[i])  
class_weights = torch.FloatTensor(weights).to(device) 
print(weights)
criterion,img_devil = nn.CrossEntropyLoss(weight=class_weights),0
fixed_noise = torch.randn(1, nz, 1, 1, device=device)

optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
schedulerD=torch.optim.lr_scheduler.ExponentialLR(optimizerD, gamma=0.86)

Unique: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14] [  63   13 1890  167    6 1793  414   39 1512  628  130 1208  495  190
 1281]
[30.0, 145.3846153846154, 1.0, 11.317365269461078, 315.0, 1.0540992749581706, 4.565217391304348, 48.46153846153846, 1.25, 3.0095541401273884, 14.538461538461538, 1.564569536423841, 3.8181818181818183, 9.947368421052632, 1.4754098360655739]


In [46]:
z_w_=[]
while(True):
    z_,z,z_w=0,0,{}
    for i, data in enumerate(dataloader, 0):
        optimizerD.zero_grad()
        real_cpu=data[0].to(device)
        label=data[1].to(device)
        output = netD(real_cpu)
        errD_real = criterion(output,label)
        errD_real.backward()
        optimizerD.step()
        wolf_z=torch.argmax(netD(real_cpu),dim=1).view(-1)
        #print(wolf_z,label)
        for i,j in zip(wolf_z,label):
            if i==j:
                if dataloader.dataset.classes[i] not in z_w.keys():
                    z_w[dataloader.dataset.classes[i]]=1
                else:
                    z_w[dataloader.dataset.classes[i]]+=1
                z+=1
            z_+=1
    z_w_.append(z)
    if len(z_w_)>=3:
        if len([True for i in range(1,3) if z_w_[len(z_w_)-i]<=z_w_[len(z_w_)-3]+2 and z_w_[len(z_w_)-i]>=z_w_[len(z_w_)-4]-3])==2:
            z_w_=[]
            print(optimizerD.param_groups[0]["lr"])
            schedulerD.step()
            print(optimizerD.param_groups[0]["lr"])
    torch.save(netD.state_dict(),f'/kaggle/working/weight_1/{z}.pth')
    torch.save(netD.state_dict(),f'/kaggle/working/rafire.pth')
    print(z_,z)	
    for i in z_w.keys():
        print(f"{i} : {z_w[i]}")

RuntimeError: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility